# Multimodal Classification of Disaster Data with GNN
## Paper Reproduction Notebook

**Paper:** *Multimodal Classification of Social Media Disaster Posts With Graph Neural Networks and Few-Shot Learning*  
**Authors:** José Nascimento, Paolo Bestagini, Anderson Rocha  
**Repository:** [jdnascim/mm-class-for-disaster-data-with-gnn](https://github.com/jdnascim/mm-class-for-disaster-data-with-gnn)

This notebook reproduces the paper's experiments by:
1. Cloning the **original repository** and using its code as-is
2. Running `feature_fusion.py` via subprocess (matching `exp_5.sh`)
3. Caching features (MaxViT + MPNet) to avoid re-extraction
4. Running all 4 labeled-set sizes × 10 splits = **40 experiments**
5. Aggregating results and comparing with paper's Table V

### Paper Results (Table V — F1-Score %)
| Config | 50 | 100 | 250 | 500 |
|---|---|---|---|---|


### Figure 1 — Problem Illustration


In [ ]:
# ═══════════════════════════════════════════════════════
# Figure 1 — Problem Illustration
# Social media noise filtering during crisis events
# ═══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
import os; os.makedirs('figures', exist_ok=True)

fig, ax = plt.subplots(1, 1, figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_aspect('equal')

# Title
ax.text(7, 5.7, 'Crisis Event → Social Media Stream → Filtering Pipeline',
        ha='center', va='top', fontsize=14, fontweight='bold', color='#1a237e')

# === Left: Social Media Stream (noisy) ===
stream_box = FancyBboxPatch((0.3, 1.5), 4.4, 3.5, boxstyle="round,pad=0.15",
                             facecolor='#ffebee', edgecolor='#c62828', linewidth=2)
ax.add_patch(stream_box)
ax.text(2.5, 4.75, '📱 Social Media Stream', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#c62828')

# Mix of informative and noise
posts = [
    ('🏚️ Building collapsed on Main St', '#c8e6c9', '✓'),
    ('😂 Funny meme about weather', '#ffcdd2', '✗'),
    ('🆘 People trapped, need rescue', '#c8e6c9', '✓'),
    ('🗳️ Political opinion tweet', '#ffcdd2', '✗'),
    ('🌊 Flood waters rising at 5th Ave', '#c8e6c9', '✓'),
    ('🎵 New song dropped today!', '#ffcdd2', '✗'),
]
for idx, (text, color, mark) in enumerate(posts):
    y = 4.2 - idx * 0.43
    box = FancyBboxPatch((0.5, y-0.15), 4.0, 0.35,
                          boxstyle="round,pad=0.05",
                          facecolor=color, edgecolor='#888', linewidth=0.5)
    ax.add_patch(box)
    ax.text(0.7, y, f'{mark} {text}', ha='left', va='center', fontsize=7.5)

# === Arrow ===
ax.annotate('', xy=(5.5, 3.25), xytext=(4.9, 3.25),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))
ax.text(5.2, 3.6, 'Filter', ha='center', fontsize=9, fontweight='bold', color='#333')

# === Middle: Model ===
model_box = FancyBboxPatch((5.5, 2.2), 3.0, 2.1, boxstyle="round,pad=0.15",
                            facecolor='#e3f2fd', edgecolor='#1565c0', linewidth=2)
ax.add_patch(model_box)
ax.text(7.0, 4.0, '🤖 GNN Classifier', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#1565c0')
ax.text(7.0, 3.5, 'CLIP Features', ha='center', fontsize=9, color='#1565c0')
ax.text(7.0, 3.1, '+ kNN Graph', ha='center', fontsize=9, color='#1565c0')
ax.text(7.0, 2.7, '+ GraphSAGE', ha='center', fontsize=9, color='#1565c0')

# === Arrow ===
ax.annotate('', xy=(9.3, 3.25), xytext=(8.7, 3.25),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))

# === Right: Filtered Output ===
out_box = FancyBboxPatch((9.3, 2.0), 4.4, 2.5, boxstyle="round,pad=0.15",
                          facecolor='#e8f5e9', edgecolor='#2e7d32', linewidth=2)
ax.add_patch(out_box)
ax.text(11.5, 4.2, '✅ Informative Posts', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#2e7d32')

filtered = [
    '🏚️ Building collapsed on Main St',
    '🆘 People trapped, need rescue',
    '🌊 Flood waters rising at 5th Ave',
]
for idx, text in enumerate(filtered):
    y = 3.7 - idx * 0.45
    box = FancyBboxPatch((9.5, y-0.15), 4.0, 0.35,
                          boxstyle="round,pad=0.05",
                          facecolor='#a5d6a7', edgecolor='#388e3c', linewidth=0.5)
    ax.add_patch(box)
    ax.text(9.7, y, text, ha='left', va='center', fontsize=8)

# === Bottom: Stats ===
ax.text(7.0, 1.3, 'Challenge: Only ~30% of crisis tweets are informative',
        ha='center', fontsize=10, style='italic', color='#666')
ax.text(7.0, 0.8, 'Goal: Few-shot learning with only 50-250 labeled samples',
        ha='center', fontsize=10, style='italic', color='#666')

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#c8e6c9', edgecolor='#388e3c', label='Informative'),
    mpatches.Patch(facecolor='#ffcdd2', edgecolor='#c62828', label='Not Informative'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig('figures/fig1_problem_illustration.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Saved: figures/fig1_problem_illustration.png")


### Figure 2 — Pipeline Architecture


In [ ]:
# ═══════════════════════════════════════════════════════
# Figure 2 — Architecture Diagram
# Full pipeline: CLIP → PCA → kNN → GraphSAGE → Classification
# ═══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import os; os.makedirs('figures', exist_ok=True)

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(-0.5, 16)
ax.set_ylim(-0.5, 8.5)
ax.axis('off')

# Title
ax.text(8, 8.2, 'Proposed Pipeline — Late Fusion + PCA + GraphSAGE (Paper Figure 2)',
        ha='center', fontsize=14, fontweight='bold', color='#1a237e')

def draw_box(ax, x, y, w, h, text, color, text_color='white', fontsize=9, subtext=None):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                          facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2 + (0.12 if subtext else 0), text,
            ha='center', va='center', fontsize=fontsize, fontweight='bold', color=text_color)
    if subtext:
        ax.text(x + w/2, y + h/2 - 0.2, subtext,
                ha='center', va='center', fontsize=7, color=text_color, alpha=0.85)

def draw_arrow(ax, x1, y1, x2, y2, text=None):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))
    if text:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx, my + 0.2, text, ha='center', fontsize=7, color='#555', style='italic')

# ═══════ Input layer ═══════
draw_box(ax, 0, 5.5, 2.2, 1.2, '📷 Image', '#e91e63', subtext='Crisis photo')
draw_box(ax, 0, 3.0, 2.2, 1.2, '📝 Text', '#9c27b0', subtext='Tweet/caption')

# ═══════ CLIP Extraction ═══════
draw_box(ax, 3.2, 5.5, 2.2, 1.2, 'CLIP Image\nEncoder', '#1565c0', subtext='ViT backbone')
draw_box(ax, 3.2, 3.0, 2.2, 1.2, 'CLIP Text\nEncoder', '#0277bd', subtext='Multilingual')

draw_arrow(ax, 2.2, 6.1, 3.2, 6.1)
draw_arrow(ax, 2.2, 3.6, 3.2, 3.6)

# Embedding dimensions
ax.text(5.6, 6.3, '512-d', fontsize=8, color='#1565c0', fontweight='bold')
ax.text(5.6, 3.8, '512-d', fontsize=8, color='#0277bd', fontweight='bold')

# ═══════ PCA Reduction ═══════
draw_box(ax, 6.2, 5.5, 1.8, 1.2, 'PCA', '#2e7d32', subtext='→ 256-d')
draw_box(ax, 6.2, 3.0, 1.8, 1.2, 'PCA', '#388e3c', subtext='→ 256-d')

draw_arrow(ax, 5.4, 6.1, 6.2, 6.1)
draw_arrow(ax, 5.4, 3.6, 6.2, 3.6)

# ═══════ Feature Combination (kNN graph) ═══════
draw_box(ax, 8.7, 4.0, 2.0, 1.8, 'Concatenate\n+ kNN Graph', '#f57f17',
         text_color='white', subtext='k=16, cosine sim')

draw_arrow(ax, 8.0, 6.1, 8.7, 5.3, 'img_feat')
draw_arrow(ax, 8.0, 3.6, 8.7, 4.5, 'txt_feat')

ax.text(9.7, 3.7, '512-d concat\n→ kNN graph', fontsize=7, ha='center',
        color='#e65100', style='italic')

# ═══════ Late Fusion GNN ═══════
draw_box(ax, 11.5, 5.5, 2.0, 1.2, 'GNN Branch\n(Image)', '#c62828',
         subtext='2×SAGE 1024')
draw_box(ax, 11.5, 3.0, 2.0, 1.2, 'GNN Branch\n(Text)', '#ad1457',
         subtext='2×SAGE 1024')

draw_arrow(ax, 10.7, 5.5, 11.5, 6.1, 'img nodes')
draw_arrow(ax, 10.7, 4.3, 11.5, 3.6, 'txt nodes')

# ═══════ Merge + Classifier ═══════
draw_box(ax, 14.0, 4.0, 1.8, 1.8, 'Merge +\nClassifier', '#1b5e20',
         subtext='Softmax')

draw_arrow(ax, 13.5, 6.1, 14.0, 5.3, 'h_img')
draw_arrow(ax, 13.5, 3.6, 14.0, 4.5, 'h_txt')

# ═══════ Output ═══════
ax.text(14.9, 3.5, '→ Informative\n  / Not-Informative', fontsize=10,
        fontweight='bold', color='#1b5e20')

# ═══════ Section labels ═══════
for x, label in [(1.1, 'Input'), (4.3, 'Feature\nExtraction'),
                  (7.1, 'Dimensionality\nReduction'), (9.7, 'Graph\nConstruction'),
                  (12.5, 'Late Fusion\nGNN'), (14.9, 'Output')]:
    ax.text(x, 2.3, label, ha='center', fontsize=8, color='#666', style='italic')

# ═══════ Fusion Method Legend ═══════
legend_text = (
    "Fusion Methods (Table 2):\n"
    "• Early: concat → 2 GNN layers → linear\n"
    "• Middle: 2 parallel GNNs → concat → 1 GNN\n"
    "• Late (shown): 2 parallel GNNs → merge → classify ★ Best"
)
props = dict(boxstyle='round,pad=0.5', facecolor='#f5f5f5', edgecolor='#ccc', alpha=0.9)
ax.text(0.5, 1.2, legend_text, fontsize=8, va='top', bbox=props,
        family='monospace')

plt.tight_layout()
plt.savefig('figures/fig2_architecture.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Saved: figures/fig2_architecture.png")


## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
import os, sys, time

# ═══════════════════════════════════════════════════════
# 1a. Clone the original repository
# ═══════════════════════════════════════════════════════
REPO = 'mm-class-for-disaster-data-with-gnn'
REPO_URL = f'https://github.com/jdnascim/{REPO}.git'

if not os.path.isdir(REPO):
    !git clone --depth 1 {REPO_URL}
    print(f'Cloned {REPO}')
else:
    print(f'Repo already exists')

%cd {REPO}
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ═══════════════════════════════════════════════════════
# AUTO-LOGGER: Captures all stages for reporting
# ═══════════════════════════════════════════════════════
import datetime, json as _json

class ExperimentLogger:
    def __init__(self):
        self.start_time = datetime.datetime.now()
        self.stages = []
        self.metadata = {}

    def log_stage(self, name, data):
        entry = {
            'stage': name,
            'timestamp': datetime.datetime.now().isoformat(),
            'elapsed_s': (datetime.datetime.now() - self.start_time).total_seconds(),
            **data
        }
        self.stages.append(entry)
        self._auto_save()
        print(f'[LOG] {name}: {data}')

    def set_meta(self, key, value):
        self.metadata[key] = value

    def _auto_save(self):
        try:
            self.save(f'experiment_log_latest.json')
        except: pass

    def save(self, path='experiment_report.json'):
        report = {
            'metadata': self.metadata,
            'start_time': self.start_time.isoformat(),
            'end_time': datetime.datetime.now().isoformat(),
            'total_time_s': (datetime.datetime.now() - self.start_time).total_seconds(),
            'stages': self.stages
        }
        with open(path, 'w') as f:
            _json.dump(report, f, indent=2, ensure_ascii=False, default=str)
        print(f'Report saved: {path}')
        return report

    def to_markdown(self):
        lines = []
        lines.append(f'# Experiment Report')
        lines.append(f'**Mode:** {self.metadata.get("experiment_mode", "N/A")}')
        lines.append(f'**Start:** {self.start_time.strftime("%Y-%m-%d %H:%M:%S")}')
        lines.append(f'**Total time:** {(datetime.datetime.now()-self.start_time).total_seconds()/60:.1f} min')
        lines.append('')
        for s in self.stages:
            lines.append(f'## {s["stage"]}')
            lines.append(f'*Elapsed: {s["elapsed_s"]/60:.1f} min*')
            for k, v in s.items():
                if k not in ('stage', 'timestamp', 'elapsed_s'):
                    lines.append(f'- **{k}:** {v}')
            lines.append('')
        return '\n'.join(lines)

LOG = ExperimentLogger()
print('Experiment logger initialized')


In [ ]:
# ═══════════════════════════════════════════════════════
# 1b. Verify GPU & PyTorch
# ═══════════════════════════════════════════════════════
import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
    print('cuDNN benchmark enabled')
else:
    print('WARNING: No GPU detected! Experiments will be very slow.')
# -- Log GPU info --
if torch.cuda.is_available():
    LOG.log_stage('GPU Setup', {
        'gpu_name': torch.cuda.get_device_name(0),
        'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        'pytorch': torch.__version__,
        'cuda': torch.version.cuda
    })


In [ ]:
# ===============================================================
# 1c. Install dependencies (optimized for Colab)
# ===============================================================

t0 = time.time()
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

# PyG (2.4+ works WITHOUT torch-scatter/torch-sparse on PyTorch 2.x)
!pip install -q torch-geometric
print(f'[1/2] PyG ({time.time()-t0:.0f}s)')

# All other deps in one call
t1 = time.time()
!pip install -q numpy pandas scikit-learn tqdm matplotlib pillow scikit-image requests umap-learn timm sentence-transformers jsonlines psutil emoji==1.7.0
print(f'[2/2] Core + ML deps ({time.time()-t1:.0f}s)')

print(f'\nAll dependencies installed! Total: {time.time()-t0:.0f}s')

# -- Log dependencies --
import torch_geometric
LOG.log_stage('Dependencies', {
    'install_time_s': round(time.time()-t0),
    'pyg_version': torch_geometric.__version__,
    'pytorch': torch.__version__,
})


## 2. Download CrisisMMD v2.0 Dataset

In [ ]:
import tarfile, shutil

DATA_DIR = 'data'
TARGET_DIR = os.path.join(DATA_DIR, 'CrisisMMD_v2.0')
TAR_PATH = os.path.join(DATA_DIR, 'CrisisMMD_v2.0.tar.gz')
URL = 'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz'

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.isdir(TARGET_DIR):
    # Download with wget (10-50x faster than Python requests on Colab)
    if not os.path.exists(TAR_PATH) or os.path.getsize(TAR_PATH) < 1.8e9:
        print('Downloading dataset with wget...')
        !wget -q --show-progress -c -O {TAR_PATH} {URL}
    else:
        print(f'Archive exists: {os.path.getsize(TAR_PATH)/1e9:.2f} GB')

    # Extract with system tar (faster than Python tarfile module)
    print('Extracting...')
    !cd {DATA_DIR} && tar xzf CrisisMMD_v2.0.tar.gz
    print('Extraction complete!')
else:
    print(f'Dataset already exists at {TARGET_DIR}')

assert os.path.isdir(TARGET_DIR), f'Dataset not found: {TARGET_DIR}'
print(f'Dataset ready: {TARGET_DIR}')

# -- Log dataset --
import glob as _glob
_n_images = len(_glob.glob(os.path.join(TARGET_DIR, '**', '*.jpg'), recursive=True))
_n_images += len(_glob.glob(os.path.join(TARGET_DIR, '**', '*.png'), recursive=True))
LOG.log_stage('Dataset', {
    'path': TARGET_DIR,
    'num_images': _n_images,
    'size_gb': round(sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fn in os.walk(TARGET_DIR) for f in fn)/1e9, 2)
})


> **Note:** 7Sept dataset experiments are skipped — dataset not publicly available (Twitter/X ToS).


## 3. Feature Extraction (Cached)

Extract features using **three** extractors, cached to `.pkl`:  
- **MaxViT** (image) + **MPNet** (text) — repo's default  
- **CLIP ViT-B/32** (image) + **CLIP Text** — paper's baseline  

Subsequent runs use cache (~2s vs ~8min per extraction).

In [ ]:
import pickle

CACHE_DIR = 'cached_features'
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Inject CLIP functions into feature_extraction.py ──
CLIP_MARKER = '# === CLIP_SUPPORT_V2 ==='
with open('feature_extraction.py', 'r') as f:
    fe_code = f.read()

if CLIP_MARKER not in fe_code:
    # Remove old CLIP injection if present
    if '# === CLIP_SUPPORT_ADDED ===' in fe_code:
        fe_code = fe_code.split('# === CLIP_SUPPORT_ADDED ===')[0]
        print('Removed old CLIP injection (upgrading to v2)')
    CLIP_INJECT = '''
# === CLIP_SUPPORT_V2 ===
# CLIP feature extraction using sentence-transformers (Multilingual CLIP)
# Paper: "multilingual version of CLIP based on ViT"
# Model: clip-ViT-B-32-multilingual-v1 (supports 50+ languages)
# Adds: clip_image_features() and clip_text_features()

from sentence_transformers import SentenceTransformer as _ST_CLIP
from PIL import Image as _PILImage

_CLIP_IMAGE_MODEL = "clip-ViT-B-32"  # vision encoder (encodes images)
_CLIP_TEXT_MODEL  = "clip-ViT-B-32-multilingual-v1"  # multilingual text encoder

def clip_image_features(gpu, sample=False):
    """Extract image features using multilingual CLIP ViT-B/32 vision encoder.
    Uses sentence-transformers multilingual CLIP as described in the paper.
    Returns list of 3 DataFrames [train, dev, test] with columns:
        image_files, text, embeddings, labels
    Embedding dim: 512
    """
    dfs = [None, None, None]

    if torch.cuda.is_available():
        dev = "cuda:" + str(gpu)
    else:
        dev = "cpu"

    # Load multilingual CLIP model (shared embedding space for images and text)
    model = _ST_CLIP(_CLIP_IMAGE_MODEL, cache_folder=".")
    model = model.to(dev)

    batch_size = 64

    for ix, split in enumerate(("train", "dev", "test")):
        data = [obj for obj in jsonlines.open(join(DATAPATH, split + ".jsonl"))]

        image_files = []
        text = []
        labels = []

        for row in tqdm.tqdm(data, desc=f"CLIP-img loading {split}"):
            image_files.append(join(IMAGEPATH, row['image']))
            text.append(row['text'])
            labels.append(row['label'])

        all_embeddings = []

        for i in tqdm.tqdm(range(0, len(image_files), batch_size),
                           desc=f"CLIP-img encoding {split}"):
            batch_files = image_files[i:i+batch_size]
            batch_images = []

            for img_path in batch_files:
                try:
                    img = _PILImage.open(img_path).convert("RGB")
                    batch_images.append(img)
                except Exception as e:
                    print(f"  Warning: Cannot load {img_path}: {e}")
                    batch_images.append(_PILImage.new("RGB", (224, 224), (128, 128, 128)))

            if not batch_images:
                continue

            # SentenceTransformer.encode handles PIL images directly
            embs = model.encode(batch_images, batch_size=len(batch_images),
                               show_progress_bar=False, convert_to_numpy=True)
            # L2 normalize
            norms = np.linalg.norm(embs, axis=1, keepdims=True)
            norms[norms == 0] = 1
            embs = embs / norms

            all_embeddings.extend(embs.tolist())

        labels_arr = [(1, 0) if l == 'not_informative' else (0, 1) for l in labels]
        labels_arr = np.array(labels_arr)

        df = pd.DataFrame({
            "image_files": image_files,
            "text": text,
            "embeddings": [e for e in all_embeddings],
            "labels": np.argmax(labels_arr, axis=1)
        })

        dfs[ix] = df
        print(f"  CLIP image features [{split}]: {len(df)} samples, dim={len(all_embeddings[0]) if all_embeddings else '?'}")

    return dfs


def clip_text_features(gpu, sample=False):
    """Extract text features using multilingual CLIP text encoder.
    Uses sentence-transformers multilingual CLIP as described in the paper.
    Supports 50+ languages for non-English crisis tweets.
    Returns list of 3 DataFrames [train, dev, test] with columns:
        image_files, text, clean_tex, embeddings, labels
    Embedding dim: 512
    """
    dfs = [None, None, None]

    if torch.cuda.is_available():
        dev = "cuda:" + str(gpu)
    else:
        dev = "cpu"

    # Load multilingual CLIP model (text encoder supports 50+ languages)
    model = _ST_CLIP(_CLIP_TEXT_MODEL, cache_folder=".")
    model = model.to(dev)

    batch_size = 128

    for ix, split in enumerate(("train", "dev", "test")):
        data = [obj for obj in jsonlines.open(join(DATAPATH, split + ".jsonl"))]

        image_files = []
        text = []
        clean_text = []
        labels = []

        for row in tqdm.tqdm(data, desc=f"CLIP-txt loading {split}"):
            image_files.append(join(IMAGEPATH, row['image']))
            text.append(row['text'])
            clean_text.append(preprocess.pre_process(row['text'],
                              keep_hashtag=True, keep_special_symbols=True))
            labels.append(row['label'])

        # SentenceTransformer.encode handles text natively (multilingual)
        all_embeddings = model.encode(clean_text, batch_size=batch_size,
                                      show_progress_bar=True, device=dev,
                                      convert_to_numpy=True)
        # L2 normalize
        norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1
        all_embeddings = all_embeddings / norms
        all_embeddings = all_embeddings.tolist()

        labels_arr = [(1, 0) if l == 'not_informative' else (0, 1) for l in labels]
        labels_arr = np.array(labels_arr)

        df = pd.DataFrame({
            "image_files": image_files,
            "text": text,
            "clean_tex": clean_text,
            "embeddings": [e for e in all_embeddings],
            "labels": np.argmax(labels_arr, axis=1)
        })

        dfs[ix] = df
        print(f"  CLIP text features [{split}]: {len(df)} samples, dim={len(all_embeddings[0]) if all_embeddings else '?'}")

    return dfs
'''
    with open('feature_extraction.py', 'a') as f:
        f.write(CLIP_INJECT)
    print('✅ Injected CLIP functions into feature_extraction.py')
else:
    print('CLIP functions already present in feature_extraction.py')

# ── Patch feature_fusion.py for CLIP support ──
CLIP_FF_MARKER = 'clip_image'
with open('feature_fusion.py', 'r') as f:
    ff_code = f.read()

if CLIP_FF_MARKER not in ff_code:
    # Add clip_image to imageft choices
    ff_code = ff_code.replace(
        "choices=['maxvit', 'mobilenet', 'mobilevit']",
        "choices=['maxvit', 'mobilenet', 'mobilevit', 'clip_image']")
    # Add clip_text to textft choices
    ff_code = ff_code.replace(
        "choices=['mpnet']",
        "choices=['mpnet', 'clip_text']")
    # Add dispatch for clip_text
    ff_code = ff_code.replace(
        'if args_cl.textft == "mpnet":',
        'if args_cl.textft == "clip_text":\n'
        '    [df_text_train, df_text_dev, df_text_test] = clip_text_features(dev_id, args_cl.sample)\n'
        'elif args_cl.textft == "mpnet":')
    # Add dispatch for clip_image
    ff_code = ff_code.replace(
        'elif args_cl.imageft == "maxvit":',
        'elif args_cl.imageft == "clip_image":\n'
        '    [df_image_train, df_image_dev, df_image_test] = clip_image_features(dev_id, args_cl.sample)\n'
        'elif args_cl.imageft == "maxvit":')
    with open('feature_fusion.py', 'w') as f:
        f.write(ff_code)
    print('✅ Patched feature_fusion.py with CLIP dispatch')
else:
    print('feature_fusion.py already patched for CLIP')

# ── Extract & Cache all features ──
from importlib import reload
import feature_extraction
reload(feature_extraction)
from feature_extraction import (
    mpnet_features, maxvit_features,
    clip_image_features, clip_text_features
)

# ── Smart extraction: only what's needed ──
# Read EXPERIMENT_MODE early to skip unnecessary extractors
_MODE = os.environ.get('EXPERIMENT_MODE', 'clip_pca')  # default

if _MODE.startswith('clip'):
    EXTRACTORS = [
        ('clip_image_features.pkl', 'CLIP (image)',  clip_image_features),
        ('clip_text_features.pkl',  'CLIP (text)',   clip_text_features),
    ]
    print('Mode:', _MODE, '-> extracting CLIP features only')
elif _MODE == 'maxvit_mpnet':
    EXTRACTORS = [
        ('mpnet_features.pkl',      'MPNet (text)',   mpnet_features),
        ('maxvit_features.pkl',     'MaxViT (image)', maxvit_features),
    ]
    print('Mode:', _MODE, '-> extracting MaxViT+MPNet only')
else:
    EXTRACTORS = [
        ('mpnet_features.pkl',      'MPNet (text)',     mpnet_features),
        ('maxvit_features.pkl',     'MaxViT (image)',   maxvit_features),
        ('clip_image_features.pkl', 'CLIP (image)',     clip_image_features),
        ('clip_text_features.pkl',  'CLIP (text)',      clip_text_features),
    ]
    print('Mode:', _MODE, '-> extracting ALL features')

for cache_name, label, fn in EXTRACTORS:
    cache_path = os.path.join(CACHE_DIR, cache_name)
    if not os.path.exists(cache_path):
        print(f'Extracting {label}...')
        t0 = time.time()
        dfs = fn(0, sample=False)
        with open(cache_path, 'wb') as f:
            pickle.dump(dfs, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f'  Cached: {os.path.getsize(cache_path)/1e6:.1f} MB ({time.time()-t0:.0f}s)')
        # Free memory
        del dfs
        import gc; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print(f'{label} cached: {os.path.getsize(cache_path)/1e6:.1f} MB')

print('\nFeatures ready for mode:', _MODE)

# -- Log features --
import glob as _glob
_caches = _glob.glob(os.path.join(CACHE_DIR, '*.pkl'))
LOG.log_stage('Feature Extraction', {
    'mode': _MODE,
    'cached_files': [os.path.basename(c) for c in _caches],
    'total_cache_mb': round(sum(os.path.getsize(c) for c in _caches)/1e6, 1)
})


## 4. Patch feature_extraction.py for Cache Loading

Append cache-loading wrappers so `feature_fusion.py` (called via subprocess)  
automatically loads from `.pkl` files instead of re-extracting.

In [ ]:
PATCH_CODE = '''
# === CACHE_PATCH_APPLIED ===
import pickle as _pkl
import os as _os
import time as _time
_CACHE_DIR = 'cached_features'

try:
    _orig_mpnet
except NameError:
    _orig_mpnet = mpnet_features
    _orig_maxvit = maxvit_features
    _orig_clip_img = clip_image_features
    _orig_clip_txt = clip_text_features

def mpnet_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'mpnet_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached MPNet text features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_mpnet(*args, **kwargs)

def maxvit_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'maxvit_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached MaxViT image features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_maxvit(*args, **kwargs)

def clip_image_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'clip_image_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached CLIP image features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_clip_img(*args, **kwargs)

def clip_text_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'clip_text_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached CLIP text features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_clip_txt(*args, **kwargs)
'''

MARKER = '# === CACHE_PATCH_APPLIED ==='
with open('feature_extraction.py', 'r') as f:
    orig = f.read()

if MARKER not in orig:
    with open('feature_extraction.py', 'w') as f:
        f.write(orig + '\n' + PATCH_CODE)
    print('Patched feature_extraction.py with cache loading (MaxViT, MPNet, CLIP)')
else:
    print('Already patched')

### Table 1 — Hyperparameters (Paper Reference)

| Hyperparameter | Value |
|---|---|
| Training Epochs | 500 |
| Optimizer | Adam |
| Learning Rate | 1×10⁻⁵ |
| Weight Decay | 1×10⁻² |
| Batch Size | 32 |
| Loss Function | NLL |
| Pseudo-Train Fraction | 0.4 |
| Number of Neighbors | 16 |
| Dropout Rate | 0.5 |
| Reduction Output Size | 256 |
| Early Stopping Patience | 300 epochs |
| Best Model Criterion | Harmonic Mean |


## 5. Run Experiments

Configuration matches the paper's **proposed method** (`exp_5.sh`):

- **Fusion:** Late (default, paper's best) or selectable via `EXPERIMENT_MODE`
- **Features:** CLIP (ViT-L/14) — image + text embeddings
- **Reduction:** PCA to 256 dimensions
- **GNN:** GraphSAGE 2-layer with normalization + residuals
- **Labeled sizes:** 50, 100, 250, 500 samples
- **Splits:** 10 random splits per size (total 40 runs)


In [ ]:
# ═══════════════════════════════════════════════════════
# 5a. Configuration — Select experiment mode
# ═══════════════════════════════════════════════════════

# ┌──────────────────────────────────────────────────────────────┐
# │ EXPERIMENT MODE: Choose one                                  │
# │                                                              │
# │ --- Paper Table 2: Ablation Study (CLIP features) ---        │
# │  'clip_pca'            → Late + PCA   (paper's best CLIP)    │
# │  'clip_late'           → Late + None                         │
# │  'clip_late_autoenc'   → Late + Autoencoder                  │
# │  'clip_early'          → Early + None                        │
# │  'clip_early_pca'      → Early + PCA                         │
# │  'clip_early_autoenc'  → Early + Autoencoder                 │
# │  'clip_middle'         → Middle + None                       │
# │  'clip_middle_pca'     → Middle + PCA                        │
# │  'clip_middle_autoenc' → Middle + Autoencoder                │
# │                                                              │
# │ --- Repo MaxViT+MPNet (extended) ---                         │
# │  'maxvit_mpnet'        → exp_5 (repo's best config)          │
# │                                                              │

EXPERIMENT_MODE = 'clip_pca'  # ← Paper's best: Late Fusion + PCA


# ── Base hyperparameters (from Paper TABLE 1 — ground truth) ──
EPOCHS     = 500
EARLY_STOP = 300

BASE_CFG = dict(
    arch       = 'sage-2l-norm-res',  # 2-layer GraphSAGE, 1024, residual+norm
    lr         = 1e-5,                # Learning rate
    wd         = 1e-2,                # Weight decay
    do         = 0.5,                 # Dropout
    l2         = 1e-2,                # L2 regularization lambda
    frac       = 0.4,                 # Pseudo-train fraction (40%/60% split)
    knn        = 16,                  # k-nearest neighbors for graph
    img        = 'clip_image',        # Default (overridden per mode)
    txt        = 'clip_text',
    fusion     = 'late',
    reduction  = None,
    pca_red    = None,
    umap_red   = None,
    autoenc    = None,
)

# Mode-specific overrides
# EXP_IDs: 100-series = CLIP Late | 110 = CLIP Early | 120 = CLIP Middle
MODE_CONFIGS = {
    # ── Repo MaxViT+MPNet ──
    'maxvit_mpnet': dict(
        img='maxvit', txt='mpnet',
        fusion='late', reduction=None,
        exp_id=5, desc='MaxViT+MPNet Late (repo best)'
    ),

    # ── CLIP Late Fusion variants (Table 2) ──
    'clip_pca': dict(
        img='clip_image', txt='clip_text',
        fusion='late', reduction='pca', pca_red=256,
        exp_id=100, desc='CLIP Late + PCA (paper best)'
    ),
    'clip_late': dict(
        img='clip_image', txt='clip_text',
        fusion='late', reduction=None,
        exp_id=101, desc='CLIP Late + None'
    ),
    'clip_late_autoenc': dict(
        img='clip_image', txt='clip_text',
        fusion='late', reduction='autoenc', autoenc='autoenc-base',
        exp_id=102, desc='CLIP Late + Autoencoder'
    ),

    # ── CLIP Early Fusion variants (Table 2) ──
    'clip_early': dict(
        img='clip_image', txt='clip_text',
        fusion='early', reduction=None,
        exp_id=110, desc='CLIP Early + None'
    ),
    'clip_early_pca': dict(
        img='clip_image', txt='clip_text',
        fusion='early', reduction='pca', pca_red=256,
        exp_id=111, desc='CLIP Early + PCA'
    ),
    'clip_early_autoenc': dict(
        img='clip_image', txt='clip_text',
        fusion='early', reduction='autoenc', autoenc='autoenc-base',
        exp_id=112, desc='CLIP Early + Autoencoder'
    ),

    # ── CLIP Middle Fusion variants (Table 2) ──
    'clip_middle': dict(
        img='clip_image', txt='clip_text',
        fusion='middle', reduction=None,
        exp_id=120, desc='CLIP Middle + None'
    ),
    'clip_middle_pca': dict(
        img='clip_image', txt='clip_text',
        fusion='middle', reduction='pca', pca_red=256,
        exp_id=121, desc='CLIP Middle + PCA'
    ),
    'clip_middle_autoenc': dict(
        img='clip_image', txt='clip_text',
        fusion='middle', reduction='autoenc', autoenc='autoenc-base',
        exp_id=122, desc='CLIP Middle + Autoencoder'
    ),

    # ── Prior Work (WIFS 2022) — for Table 4 comparison ──
}

mode = MODE_CONFIGS[EXPERIMENT_MODE]
CFG = {**BASE_CFG, **mode}
EXP_ID = CFG.pop('exp_id')
EXP_DESC = CFG.pop('desc')

# Labeled set sizes & splits
LABELED_SIZES = [50, 100, 250, 500]
NUM_SPLITS = 10

RUNS = []
for n in LABELED_SIZES:
    for s in range(NUM_SPLITS):
        RUNS.append({'n': n, 'split': f'{n}_s{s}', 's': s})

print(f'Experiment: {EXP_DESC}')
print(f'  Mode: {EXPERIMENT_MODE} | EXP_ID: {EXP_ID}')
print(f'  Fusion: {CFG["fusion"]} | Arch: {CFG["arch"]}')
print(f'  Features: {CFG["img"]} (image) + {CFG["txt"]} (text)')
if CFG.get('reduction'):
    red_info = CFG['reduction']
    if CFG.get('pca_red'): red_info += f' (dim={CFG["pca_red"]})'
    if CFG.get('autoenc'): red_info += f' ({CFG["autoenc"]})'
    print(f'  Reduction: {red_info}')
else:
    print(f'  Reduction: None')
print(f'  Epochs: {EPOCHS} | Early Stop: {EARLY_STOP}')
print(f'  Total runs: {len(RUNS)} ({len(LABELED_SIZES)} sizes x {NUM_SPLITS} splits)')

# -- Log experiment config --
LOG.set_meta('experiment_mode', EXPERIMENT_MODE)
LOG.log_stage('Experiment Config', {
    'mode': EXPERIMENT_MODE,
    'fusion': CFG.get('fusion'),
    'arch': CFG.get('arch'),
    'img_features': CFG.get('img'),
    'txt_features': CFG.get('txt'),
    'reduction': CFG.get('reduction'),
    'epochs': EPOCHS,
    'early_stop': EARLY_STOP,
    'labeled_sizes': LABELED_SIZES,
    'num_splits': NUM_SPLITS,
    'total_runs': len(RUNS)
})

In [ ]:
# ═══════════════════════════════════════════════════════
# 5b. Clean old results (optional — only for fresh run)
# ═══════════════════════════════════════════════════════
import glob

# Uncomment the next 3 lines to clear previous results:
# if os.path.isdir('results'):
#     shutil.rmtree('results')
#     print('Removed old results')

existing = glob.glob('results/**/*.json', recursive=True)
print(f'Existing result files: {len(existing)}')

In [ ]:
import json
# ═══════════════════════════════════════════════════════
# 5c. Run all experiments via subprocess
#     (supports MaxViT/MPNet, CLIP, PCA, etc.)
#     + Runtime tracking for paper Tables 2 & 4
# ═══════════════════════════════════════════════════════
import subprocess

IP = './data/CrisisMMD_v2.0/'
RUN_ID = 0

total_t0 = time.time()
completed = 0
failed = 0

# Runtime log: records per-run timing for paper comparison
RUNTIME_LOG_FILE = 'runtime_log.json'
runtime_log = {}
if os.path.exists(RUNTIME_LOG_FILE):
    with open(RUNTIME_LOG_FILE) as f:
        content = f.read().strip()
        runtime_log = json.loads(content) if content else {}
runtime_key = f'exp_{EXP_ID}'
if runtime_key not in runtime_log:
    runtime_log[runtime_key] = {'runs': [], 'config': EXP_DESC}

for i, r in enumerate(RUNS, 1):
    # Skip if result already exists
    result_path = f'results/CrisisMMD/gnn/{EXP_ID}/{r["n"]}_s{r["s"]}_{RUN_ID}.json'
    if os.path.exists(result_path):
        print(f'[{i}/{len(RUNS)}] SKIP {r["split"]} (result exists)')
        completed += 1
        continue

    # Build command
    cmd = (
        f'"{sys.executable}" feature_fusion.py'
        f' --gpu_id 0'
        f' --epochs {EPOCHS}'
        f' --lr {CFG["lr"]}'
        f' --weight_decay {CFG["wd"]}'
        f' --n_neigh_train {CFG["knn"]}'
        f' --n_neigh_full {CFG["knn"]}'
        f' --lbl_train_frac {CFG["frac"]}'
        f' --imagepath {IP}'
        f' --datasplit {r["split"]}'
        f' --reg l2'
        f' --l2_lambda {CFG["l2"]}'
        f' --exp_id {EXP_ID}'
        f' --dropout {CFG["do"]}'
        f' --arch {CFG["arch"]}'
        f' --loss nll'
        f' --shuffle_split'
        f' --imageft {CFG["img"]}'
        f' --textft {CFG["txt"]}'
        f' --fusion {CFG["fusion"]}'
        f' --early_stopping {EARLY_STOP}'
        f' --best_model best_hm'
        f' --run_id {RUN_ID}'
    )

    # Add optional reduction flags
    if CFG.get('reduction'):
        cmd += f' --reduction {CFG["reduction"]}'
        if CFG['reduction'] == 'pca' and CFG.get('pca_red'):
            cmd += f' --pca_red {CFG["pca_red"]}'
        elif CFG['reduction'] == 'umap' and CFG.get('umap_red'):
            cmd += f' --umap_red {CFG["umap_red"]}'
        elif CFG['reduction'] == 'autoenc' and CFG.get('autoenc'):
            cmd += f' --autoenc {CFG["autoenc"]}'

    print(f'\n[{i}/{len(RUNS)}] Running: {r["split"]}')
    t0 = time.time()
    try:
        p = subprocess.run(cmd, shell=True, timeout=7200)
        ok = (p.returncode == 0)
    except subprocess.TimeoutExpired:
        print(f'  TIMEOUT after 2h'); ok = False
    except Exception as e:
        print(f'  ERROR: {e}'); ok = False

    elapsed = time.time() - t0
    status = 'OK' if ok else 'FAIL'
    print(f'  -> {status} ({elapsed:.0f}s)')
    
    # Log runtime
    runtime_log[runtime_key]['runs'].append({
        'split': r['split'],
        'n_labeled': r['n'],
        'split_id': r['s'],
        'elapsed_s': round(elapsed, 1),
        'status': status
    })
    
    if ok:
        completed += 1
    else:
        failed += 1

total_time = time.time() - total_t0

# Save runtime log
runtime_log[runtime_key]['total_time_s'] = round(total_time, 1)
runtime_log[runtime_key]['total_time_h'] = round(total_time / 3600, 2)
with open(RUNTIME_LOG_FILE, 'w') as f:
    json.dump(runtime_log, f, indent=2)

print(f'\n{"="*50}')
print(f'ALL RUNS COMPLETE ({EXP_DESC})')
print(f'  Completed: {completed}/{len(RUNS)}')
print(f'  Failed: {failed}')
print(f'  Total time: {total_time/3600:.1f}h')
print(f'  Runtime log saved: {RUNTIME_LOG_FILE}')


## 6. Results — Aggregation & Paper Comparison

In [ ]:
# ═══════════════════════════════════════════════════════
# 6a. Load & aggregate results (all experiments)
#     + Runtime display (Paper Tables 2 & 4)
# ═══════════════════════════════════════════════════════
import json, glob, numpy as np, os
from collections import defaultdict
from IPython.display import display, HTML

# Map exp_id → experiment name
EXP_ID_MAP = {
    5:   'MaxViT+MPNet Late',
    100: 'CLIP Late+PCA',
    101: 'CLIP Late',
    102: 'CLIP Late+AE',
    110: 'CLIP Early',
    111: 'CLIP Early+PCA',
    112: 'CLIP Early+AE',
    120: 'CLIP Middle',
    121: 'CLIP Middle+PCA',
    122: 'CLIP Middle+AE',
}

# Load ALL result files
all_results = {}
for f in glob.glob('results/**/*.json', recursive=True):
    try:
        with open(f) as fp:
            d = json.load(fp)
        d['_file'] = f
        parts = f.replace('\\','/').split('/')
        exp_id = int(parts[-2])
        fname = os.path.basename(f)
        size = int(fname.split('_')[0])
        if exp_id not in all_results:
            all_results[exp_id] = defaultdict(list)
        all_results[exp_id][size].append(d)
    except:
        pass

total_files = sum(sum(len(v) for v in sizes.values()) for sizes in all_results.values())
print(f'Loaded {total_files} result files across {len(all_results)} experiments')
for eid in sorted(all_results.keys()):
    name = EXP_ID_MAP.get(eid, f'exp_{eid}')
    n_files = sum(len(v) for v in all_results[eid].values())
    sizes = sorted(all_results[eid].keys())
    print(f'  exp_{eid} ({name}): {n_files} files, sizes={sizes}')

# Load runtime log
RUNTIME_LOG_FILE = 'runtime_log.json'
runtime_log = {}
if os.path.exists(RUNTIME_LOG_FILE):
    with open(RUNTIME_LOG_FILE) as f:
        content = f.read().strip()
        runtime_log = json.loads(content) if content else {}
    print(f'\nRuntime log loaded: {len(runtime_log)} experiments')
    for key, val in runtime_log.items():
        n_runs = len(val.get('runs', []))
        total_h = val.get('total_time_h', 0)
        print(f'  {key}: {n_runs} runs, {total_h:.1f}h total')

# Paper comparison values (CLIP-based, paper Table V)
# ── Paper TABLE 4: "Ours" on CrisisMMD (F1-Score %) ──
# Values: mean ± std from paper
paper_f1 = {
    50:  (74.8, 2.0),   # 74.8 ± 2.0
    100: (76.3, 0.7),   # 76.3 ± 0.7
    250: (77.1, 0.4),   # 77.1 ± 0.4
}
# ── Paper TABLE 4: Baselines on CrisisMMD (F1-Score %) ──
sirbu_eda = {
    50:  (60.5, 5.1),   # [7]-EDA
    100: (66.1, 2.9),
    250: (71.5, 1.4),
}
sirbu_bt = {
    50:  (62.8, 5.9),   # [7]-BT
    100: (66.9, 4.2),
    250: (71.9, 1.9),
}
nascimento = {
    50:  (64.5, 5.1),   # [9] (prior work)
    100: (68.2, 3.0),
    250: (72.2, 1.3),
}

# ── Paper TABLE 4: Runtime (seconds) ──
paper_runtime = {
    'eda': (8458, 1122),
    'bt':  (7661, 267),
    'prior': (317, 9),
    'ours': (179, 17),
}

# ── Paper TABLE 3: Sirbu reproduction (F1-Score %) ──
sirbu_reproduction = {
    'eda': {250: 71.5, 500: 72.8},  # Repr.
    'bt':  {250: 71.9, 500: 72.0},
    'eda_ref': {250: 70.1, 500: 75.6},  # Ref.
    'bt_ref':  {250: 74.3, 500: 76.0},
}

# ── Paper TABLE 5: Single-modality F1-Score (%) ──
single_modality_f1 = {
    'CrisisMMD': {
        ('No', 'No'):   (73.3, 1.9),
        ('No', 'Yes'):  (76.9, 1.7),
        ('Yes', 'No'):  (74.8, 4.2),
        ('Yes', 'Yes'): (75.4, 2.1),
    },
    '7Sept': {
        ('No', 'No'):   (94.4, 1.5),
        ('No', 'Yes'):  (63.0, 6.6),
        ('Yes', 'No'):  (84.4, 6.9),
        ('Yes', 'Yes'): (95.2, 2.3),
    },
}

# Legacy alias for backward compatibility
sirbu_f1 = sirbu_eda


In [ ]:
# ═══════════════════════════════════════════════════════
# 6b. Comparison Plot — Reproduction vs Paper (Table 4)
# ═══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np
import glob, json as _json, os as _os
from collections import defaultdict

# ── 1. Find ALL result JSON files (CWD-agnostic) ──
_REPO_ROOT = _os.path.dirname(_os.path.abspath(__file__)) if '__file__' in dir() else None
_CWD = _os.getcwd()

def _collect_results():
    """Find all result JSON files by scanning known paths."""
    search_roots = [
        _CWD,
        '/content/mm-class-for-disaster-data-with-gnn',
        '/content',
        _os.path.expanduser('~'),
    ]
    for root in search_roots:
        pattern = _os.path.join(root, 'results', '**', '*.json')
        files = glob.glob(pattern, recursive=True)
        if files:
            print(f"Found {len(files)} result files under: {root}/results/")
            return files
    # deep scan fallback
    files = glob.glob('/content/**/results/**/*.json', recursive=True)
    if files:
        print(f"Found {len(files)} result files (deep scan)")
    return files

all_json_files = _collect_results()
print(f"CWD: {_CWD}")
print(f"Total result files: {len(all_json_files)}")
if all_json_files:
    print(f"Sample: {all_json_files[0]}")

# ── 2. Parse and aggregate by exp_id and labeled size ──
EXP_ID_MAP = {
    5:   'MaxViT+MPNet Late',
    100: 'CLIP Late+PCA',
    101: 'CLIP Late',
    102: 'CLIP Late+AE',
    110: 'CLIP Early',
    111: 'CLIP Early+PCA',
    112: 'CLIP Early+AE',
    120: 'CLIP Middle',
    121: 'CLIP Middle+PCA',
    122: 'CLIP Middle+AE',
}

all_results = {}
parse_errors = 0
for _f in all_json_files:
    try:
        with open(_f) as _fp:
            _d = _json.load(_fp)
        _d['_file'] = _f
        # exp_id is second-to-last path component: .../results/CrisisMMD/gnn/{exp_id}/{file}
        _parts = _f.replace('\\', '/').split('/')
        try:
            _eid = int(_parts[-2])
        except (ValueError, IndexError):
            continue
        _fname = _os.path.basename(_f)
        try:
            _size = int(_fname.split('_')[0])
        except (ValueError, IndexError):
            continue
        if _eid not in all_results:
            all_results[_eid] = defaultdict(list)
        all_results[_eid][_size].append(_d)
    except Exception:
        parse_errors += 1

total_files = sum(sum(len(v) for v in s.values()) for s in all_results.values())
print(f"\nParsed: {total_files} results across {len(all_results)} experiments ({parse_errors} errors)")
for _eid in sorted(all_results.keys()):
    _name = EXP_ID_MAP.get(_eid, f'exp_{_eid}')
    _nf = sum(len(v) for v in all_results[_eid].values())
    _sz = sorted(all_results[_eid].keys())
    print(f"  exp_{_eid} ({_name}): {_nf} files, sizes={_sz}")

# ── 3. Pick best experiment to show ──
_PREFERRED = [100, 5, 101, 102, 110, 111, 112, 120, 121, 122]
_best_eid = next((e for e in _PREFERRED if e in all_results), None)
if _best_eid is None and all_results:
    _best_eid = sorted(all_results.keys())[0]

groups = all_results.get(_best_eid, {}) if _best_eid is not None else {}
_exp_name = EXP_ID_MAP.get(_best_eid, f'exp_{_best_eid}') if _best_eid else 'None'
print(f"\nPlotting: {_exp_name} (exp_id={_best_eid}), sizes={sorted(groups.keys())}")

# ── 4. Paper reference values ──
try:
    paper_f1
except NameError:
    paper_f1 = {50: (74.8, 2.0), 100: (76.3, 0.7), 250: (77.1, 0.4)}
try:
    sirbu_eda
except NameError:
    sirbu_eda  = {50: (60.5, 5.1), 100: (66.1, 2.9), 250: (71.5, 1.4)}
try:
    sirbu_bt
except NameError:
    sirbu_bt   = {50: (62.8, 5.9), 100: (66.9, 4.2), 250: (71.9, 1.9)}
try:
    nascimento
except NameError:
    nascimento = {50: (64.5, 5.1), 100: (68.2, 3.0), 250: (72.2, 1.3)}

# ── 5. Plot ──
if not groups:
    print("\nNo results to plot.")
    print(f"Expected: results/CrisisMMD/gnn/<exp_id>/*.json")
    print(f"CWD: {_CWD}")
else:
    fig, ax = plt.subplots(figsize=(12, 7))

    # Our reproduction
    our_S, our_means, our_stds = [], [], []
    for s in sorted(groups.keys()):
        f1s = [r.get('f1_test', r.get('test_f1', 0)) * 100 for r in groups[s]]
        f1s = [x for x in f1s if x > 0]
        if f1s:
            our_S.append(s)
            our_means.append(np.mean(f1s))
            our_stds.append(np.std(f1s))

    if our_S:
        ax.errorbar(our_S, our_means, yerr=our_stds,
                    fmt='-D', lw=2.5, ms=9, capsize=5,
                    label=f'Our Reproduction ({_exp_name})',
                    color='#1565C0', zorder=5)
        for x, y, s in zip(our_S, our_means, our_stds):
            ax.annotate(f'{y:.1f}\u00b1{s:.1f}', (x, y),
                        textcoords="offset points", xytext=(0, 16),
                        ha='center', fontsize=9, fontweight='bold', color='#1565C0')

    # Paper "Ours"
    if paper_f1:
        p_S = sorted(paper_f1.keys())
        ax.errorbar(p_S, [paper_f1[s][0] for s in p_S],
                    yerr=[paper_f1[s][1] for s in p_S],
                    fmt='--s', lw=2, ms=8, capsize=4,
                    label='Paper — Ours (Table 4)', color='#E65100', zorder=4)
        for s in p_S:
            pm, ps = paper_f1[s]
            ax.annotate(f'{pm:.1f}\u00b1{ps:.1f}', (s, pm),
                        textcoords="offset points", xytext=(0, -18),
                        ha='center', fontsize=8, color='#E65100')

    # Baselines
    for bdata, blabel, bfmt, bcolor in [
        (sirbu_eda,  '[7]-EDA', ':^', '#66BB6A'),
        (sirbu_bt,   '[7]-BT',  ':v', '#AB47BC'),
        (nascimento, '[9]',     ':o', '#EF5350'),
    ]:
        if bdata:
            b_S = sorted(bdata.keys())
            ax.errorbar(b_S, [bdata[s][0] for s in b_S],
                        yerr=[bdata[s][1] for s in b_S],
                        fmt=bfmt, lw=1.5, ms=6, capsize=3,
                        label=f'Paper — {blabel} (Table 4)',
                        color=bcolor, alpha=0.8)

    ax.set_xlabel('Number of Labeled Samples', fontsize=13)
    ax.set_ylabel('F1-Score (%)', fontsize=13)
    ax.set_title(f'CrisisMMD — Few-Shot Classification\nReproduction vs Paper (Table 4)\n({_exp_name})', fontsize=13)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xticks([50, 100, 250, 500])
    ax.set_ylim(50, 90)
    plt.tight_layout()
    _os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/reproduction_vs_paper.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/reproduction_vs_paper.png')

    # Summary table
    print(f'\n{"="*60}')
    print(f'{"Labeled":>8} | {"Ours (mean\u00b1std)":>16} | {"Paper":>14} | {"Gap":>6}')
    print(f'{"-"*8}-+-{"-"*16}-+-{"-"*14}-+-{"-"*6}')
    for s, m, sd in zip(our_S, our_means, our_stds):
        ours_str = f'{m:.1f}\u00b1{sd:.1f}%'
        paper_str = f'{paper_f1[s][0]:.1f}\u00b1{paper_f1[s][1]:.1f}%' if s in paper_f1 else '--'
        gap = f'{m - paper_f1[s][0]:+.1f}' if s in paper_f1 else '--'
        print(f'{s:>8} | {ours_str:>16} | {paper_str:>14} | {gap:>6}')


### 6c. Table 2 — Ablation Study Results (CrisisMMD)


In [ ]:
# ═══════════════════════════════════════════════════════
# 6c. Paper Table 2 — Proposed Method Results
# ═══════════════════════════════════════════════════════
import json, glob, os
import numpy as np
from IPython.display import display, HTML
from collections import defaultdict

# ── Find results dir (CWD-agnostic) ──
def _find_result_files():
    candidates = [
        os.getcwd(),
        '/content/mm-class-for-disaster-data-with-gnn',
        '/content',
    ]
    for base in candidates:
        # Recursive search: results/CrisisMMD/gnn/{exp_id}/*.json
        pattern = os.path.join(base, 'results', '**', '*.json')
        files = glob.glob(pattern, recursive=True)
        if files:
            print(f"Found {len(files)} result files under: {base}/results/")
            return files
    # deep scan fallback
    files = glob.glob('/content/**/results/**/*.json', recursive=True)
    if files:
        print(f"Found {len(files)} result files (deep scan)")
    return files

all_json_files = _find_result_files()

# Parse: group by labeled_size (use exp_id 100 = clip_pca as primary)
# Also detect exp_id from path
all_results = defaultdict(list)  # n_labeled -> list of result dicts
exp_id_used = None

for f in sorted(all_json_files):
    try:
        with open(f) as fp:
            d = json.load(fp)
        d['_file'] = f

        # Get exp_id from path second-to-last component
        parts = f.replace('\\', '/').split('/')
        try:
            file_eid = int(parts[-2])
            d['_exp_id'] = file_eid
        except (ValueError, IndexError):
            file_eid = None

        # Get labeled size from filename: 250_s4_0.json -> 250
        fname = os.path.basename(f)
        try:
            n = int(fname.split('_')[0])
            d['n_labeled'] = n
        except (ValueError, IndexError):
            n = d.get('n_labeled', 0)

        # Only aggregate the "best" exp (prefer 100=clip_pca)
        if exp_id_used is None:
            exp_id_used = file_eid
        if file_eid == exp_id_used or file_eid is None:
            if n > 0:
                all_results[n].append(d)
    except Exception:
        pass

EXP_NAME_MAP = {
    5: 'MaxViT+MPNet Late', 100: 'CLIP Late+PCA', 101: 'CLIP Late',
    102: 'CLIP Late+AE', 110: 'CLIP Early', 111: 'CLIP Early+PCA',
}
exp_display = EXP_NAME_MAP.get(exp_id_used, f'exp_{exp_id_used}')

if not all_results:
    print('No results found.')
    print(f'CWD: {os.getcwd()}')
    print('Expected files at: results/CrisisMMD/gnn/<exp_id>/<n>_s<seed>_0.json')
else:
    # Build HTML table
    h = f'<h3>Table 2 — Proposed Method: GNN + CLIP (CrisisMMD)</h3>'
    h += f'<p>Reproduction results using <code>{exp_display}</code> (exp_id={exp_id_used})</p>'
    h += '<table border=1 cellpadding=8 style="border-collapse:collapse; font-size:14px;">'
    h += '<tr style="background:#1a237e; color:white;">'
    h += '<th>Labeled</th><th>Splits</th><th>F1 (mean ± std)</th><th>Min</th><th>Max</th>'
    
    # Paper reference column
    PAPER_REF = {50: (74.8, 2.0), 100: (76.3, 0.7), 250: (77.1, 0.4)}
    h += '<th>Paper (Table 4)</th><th>Gap</th></tr>'
    
    overall_f1s = []
    for n_labeled in sorted(all_results.keys()):
        runs = all_results[n_labeled]
        f1s = []
        for r in runs:
            f1 = r.get('f1_test', r.get('test_f1', 0))
            if f1 > 0:
                if f1 < 1:
                    f1 *= 100
                f1s.append(f1)
        
        if f1s:
            m, sd = np.mean(f1s), np.std(f1s)
            overall_f1s.extend(f1s)
            
            # Paper ref
            if n_labeled in PAPER_REF:
                pm, ps = PAPER_REF[n_labeled]
                paper_str = f'{pm:.1f} ± {ps:.1f}'
                gap = m - pm
                gap_color = '#2e7d32' if gap >= 0 else '#c62828'
                gap_str = f'<span style="color:{gap_color};font-weight:bold">{gap:+.1f}</span>'
            else:
                paper_str, gap_str = '—', '—'
            
            h += f'<tr style="background:#e3f2fd;">'
            h += f'<td style="text-align:center;font-weight:bold;">{n_labeled}</td>'
            h += f'<td style="text-align:center;">{len(f1s)}</td>'
            h += f'<td style="text-align:center;font-weight:bold;">{m:.2f} ± {sd:.2f}</td>'
            h += f'<td style="text-align:center;">{min(f1s):.2f}</td>'
            h += f'<td style="text-align:center;">{max(f1s):.2f}</td>'
            h += f'<td style="text-align:center;">{paper_str}</td>'
            h += f'<td style="text-align:center;">{gap_str}</td></tr>'
    
    # Overall row
    if overall_f1s:
        h += f'<tr style="background:#c5cae9; font-weight:bold;">'
        h += f'<td>Overall</td><td>{len(overall_f1s)}</td>'
        h += f'<td>{np.mean(overall_f1s):.2f} ± {np.std(overall_f1s):.2f}</td>'
        h += f'<td>{min(overall_f1s):.2f}</td><td>{max(overall_f1s):.2f}</td>'
        h += '<td colspan=2>—</td></tr>'
    
    h += '</table>'
    h += '<p><small>Config: CLIP Late Fusion + PCA (dim=256), SAGE 2-layer, kNN=16, frac=0.4, '
    h += 'lr=1e-5, wd=1e-2, dropout=0.5</small></p>'
    
    display(HTML(h))
    print(f'\nTotal runs loaded: {sum(len(v) for v in all_results.values())}')
    print(f'Label sizes found: {sorted(all_results.keys())}')
    print(f'Experiment: {exp_display} (exp_id={exp_id_used})')


### 6d. Figure 3 — Error Analysis: CrisisMMD

Most commonly mistaken instances from the test set.


In [ ]:
# ═══════════════════════════════════════════════════════
# 6d. Figure 3 — Most Commonly Mistaken Instances (CrisisMMD)
#
# Paper caption: "Most commonly mistaken instances from the
# test set of CrisisMMD. It seems that the method tend to be
# confused if an instance presenting a landscape with disaster
# patterns is not informative."
# ═══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os, glob, json, re
from collections import defaultdict
from IPython.display import display
import textwrap

# ─── Utilities ─────────────────────────────────────────
def _search_roots():
    return [os.getcwd(),
            '/content/mm-class-for-disaster-data-with-gnn',
            '/content']

# Install ftfy for robust mojibake fixing (handles emoji + all encodings)
try:
    import ftfy as _ftfy
    _HAS_FTFY = True
except ImportError:
    import subprocess as _sp
    _sp.run(['pip', 'install', 'ftfy', '-q'], capture_output=True)
    try:
        import ftfy as _ftfy
        _HAS_FTFY = True
    except ImportError:
        _HAS_FTFY = False

def _fix_text(t):
    """Fix encoding issues: mojibake (cp1252/utf-8 mix) + control chars."""
    if not isinstance(t, str):
        return t
    if _HAS_FTFY:
        t = _ftfy.fix_text(t)
    else:
        # Fallback: manual cp1252 reverse (handles most common cases)
        try:
            t = t.encode('cp1252').decode('utf-8')
        except Exception:
            pass
    # Strip C0/C1 control characters that break matplotlib fonts
    t = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', t)
    return t

# ─── Step 1: Find result files (CWD-agnostic) ──────────
def _find_result_files():
    for base in _search_roots():
        files = glob.glob(os.path.join(base, 'results', '**', '*.json'), recursive=True)
        if files:
            print(f'Found {len(files)} result files under: {base}')
            return base, files
    files = glob.glob('/content/**/results/**/*.json', recursive=True)
    return '/content', files

repo_base, result_files = _find_result_files()

results_with_preds, results_all = [], []
exp_id_used = None

for rf in sorted(result_files):
    try:
        with open(rf, 'r') as fh:
            data = json.load(fh)
        data['_file'] = rf
        parts = rf.replace('\\', '/').split('/')
        try:
            eid = int(parts[-2])
            data['_exp_id'] = eid
            if exp_id_used is None:
                exp_id_used = eid
        except (ValueError, IndexError):
            pass
        if data.get('_exp_id', exp_id_used) == exp_id_used:
            if 'f1_test' in data:
                results_all.append(data)
            if 'annot' in data and 'pred' in data:
                results_with_preds.append(data)
    except Exception:
        continue

print(f'Result files: {len(results_all)} total, {len(results_with_preds)} with per-sample predictions')
print(f'Experiment ID: {exp_id_used}')

# ─── Step 2: Load test samples from dev.jsonl ──────────
def _find_dev_jsonl():
    for base in _search_roots():
        patterns = [
            os.path.join(base, 'data', 'CrisisMMD_v2.0_baseline_split',
                         'data_splits_ssl', '*', 'informative_orig', 'dev.jsonl'),
            os.path.join(base, '**', 'dev.jsonl'),
        ]
        for pat in patterns:
            files = sorted(glob.glob(pat, recursive=True))
            if files:
                return files[0]
    return None

dev_file = _find_dev_jsonl()
test_samples = []
if dev_file:
    print(f'Loading test samples from: {dev_file}')
    with open(dev_file, 'r', encoding='utf-8', errors='replace') as fh:
        for line in fh:
            line = line.strip()
            if line:
                row = json.loads(line)
                for key in ('text', 'tweet_text'):
                    if key in row:
                        row[key] = _fix_text(row[key])
                test_samples.append(row)
    print(f'Test samples loaded: {len(test_samples)}')
else:
    print('[WARN] dev.jsonl not found — predictions cannot be aligned to samples.')

# ─── Step 3: Find image directory ──────────────────────
def _find_image_dir():
    for base in _search_roots():
        for suffix in ('data/CrisisMMD_v2.0/data_image',
                       'data/CrisisMMD_v2.0_baseline_split/data_image'):
            p = os.path.join(base, suffix)
            if os.path.isdir(p):
                return p
    for p in glob.glob('/content/**/data_image', recursive=True):
        if os.path.isdir(p):
            return p
    return None

IMAGE_BASE = _find_image_dir()
print(f'Image directory: {IMAGE_BASE}')

def _resolve_image(img_rel):
    """Locate an image file.
    dev.jsonl stores: 'data_image/hurricane_harvey/.../filename.jpg'
    IMAGE_BASE already ends with '/data_image'
    → must strip leading 'data_image/' to avoid doubled prefix.
    """
    if not img_rel or not IMAGE_BASE:
        return None
    img_rel = img_rel.replace('\\', '/')
    basename = os.path.basename(img_rel)

    # Strip doubled prefix
    for pfx in ('data_image/', 'data_image\\'):
        if img_rel.startswith(pfx):
            img_rel = img_rel[len(pfx):]
            break

    candidates = [
        os.path.join(IMAGE_BASE, img_rel),   # event/subdir/filename.jpg
        os.path.join(IMAGE_BASE, basename),  # filename.jpg only
    ]
    for c in candidates:
        if os.path.isfile(c):
            return c
    # Slow recursive fallback
    hits = glob.glob(os.path.join(IMAGE_BASE, '**', basename), recursive=True)
    return hits[0] if hits else None

# ─── Step 4: Compute per-sample error rate ─────────────
if results_with_preds and test_samples:
    # Verify alignment
    test_set_sizes = set(len(r['annot']) for r in results_with_preds)
    if len(test_set_sizes) > 1:
        print(f'[WARN] Inconsistent test set sizes: {test_set_sizes}')
    else:
        print(f'[OK] All {len(results_with_preds)} splits have consistent test size: {list(test_set_sizes)[0]}')

    sample_error_count = defaultdict(int)
    sample_total_count = defaultdict(int)
    n_splits_used = 0

    for result in results_with_preds:
        y_true = np.array(result['annot'])
        y_pred = np.array(result['pred'])
        n = min(len(y_true), len(test_samples))
        n_splits_used += 1
        for i in range(n):
            sample_total_count[i] += 1
            if y_true[i] != y_pred[i]:
                sample_error_count[i] += 1

    sample_error_rate = {
        idx: sample_error_count[idx] / sample_total_count[idx] * 100
        for idx in sample_total_count
    }

    sorted_errors = sorted(sample_error_rate.items(), key=lambda x: -x[1])
    top_6 = [(idx, rate) for idx, rate in sorted_errors if rate > 0][:6]

    print(f'\nEvaluated across {n_splits_used} splits')

    # Diagnostic table
    print(f'\n{"Rank":<6} {"Idx":<7} {"Wrong":<7} {"Total":<7} {"Error%":<8}')
    print("─" * 40)
    for rank, (idx, rate) in enumerate(top_6, 1):
        w = sample_error_count[idx]
        t = sample_total_count[idx]
        print(f'{rank:<6} {idx:<7} {w:<7} {t:<7} {rate:.0f}%')

    # ─── Plot Figure 3 ─────────────────────────────────
    if top_6:
        fig, axes = plt.subplots(2, 3, figsize=(15, 12))
        fig.patch.set_facecolor('white')

        for plot_idx, (sample_idx, error_rate) in enumerate(top_6):
            row, col = plot_idx // 3, plot_idx % 3
            ax = axes[row, col]

            sample  = test_samples[sample_idx] if sample_idx < len(test_samples) else {}
            label   = sample.get('label', 'unknown')
            text    = sample.get('text', sample.get('tweet_text', ''))
            img_rel = sample.get('image', sample.get('img_path', ''))

            # Title (red, bold — matches paper)
            ax.set_title(
                f'Label: {label} | Error Rate: ({error_rate:.0f}%)',
                fontsize=12, fontweight='bold', color='#C62828', pad=10
            )
            for spine in ax.spines.values():
                spine.set_linewidth(2.5)
                spine.set_color('#C62828')

            # Image
            img_path = _resolve_image(img_rel)
            if img_path:
                try:
                    img = mpimg.imread(img_path)
                    ax.imshow(img)
                except Exception as e:
                    ax.text(0.5, 0.5, f'Load error', transform=ax.transAxes,
                            ha='center', va='center', fontsize=9, color='red')
                    ax.set_facecolor('#fff0f0')
            else:
                fname = os.path.basename(img_rel) if img_rel else 'N/A'
                ax.text(0.5, 0.5, f'Image not found:\n{fname}',
                        transform=ax.transAxes, ha='center', va='center',
                        fontsize=9, color='gray')
                ax.set_facecolor('#f5f5f5')

            ax.set_xticks([])
            ax.set_yticks([])

            # Tweet text (clean and wrapped)
            clean_text = _fix_text(text)
            wrapped = textwrap.fill(clean_text, width=45) if clean_text else 'N/A'
            if len(wrapped) > 160:
                wrapped = wrapped[:157] + '...'
            ax.set_xlabel(wrapped, fontsize=8, style='italic',
                          color='#333', labelpad=10)

        for i in range(len(top_6), 6):
            axes[i // 3, i % 3].set_visible(False)

        fig.suptitle(
            'FIGURE 3.  Most commonly mistaken instances from the test set of CrisisMMD.\n'
            'It seems that the method tend to be confused if an instance presenting\n'
            'a landscape with disaster patterns is not informative.',
            fontsize=11, fontweight='bold', y=0.02, va='bottom'
        )
        plt.tight_layout(rect=[0, 0.06, 1, 1])
        os.makedirs('figures', exist_ok=True)
        plt.savefig('figures/figure3_crisismmd.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('\nSaved: figures/figure3_crisismmd.png')

        print(f'\n{"Rank":<5} {"Error%":<8} {"Label":<20} {"Tweet text"}')
        print("─" * 90)
        for rank, (idx, rate) in enumerate(top_6, 1):
            s   = test_samples[idx] if idx < len(test_samples) else {}
            txt = _fix_text(s.get('text', s.get('tweet_text', '?')))[:60]
            print(f'{rank:<5} {rate:>5.0f}%   {s.get("label","?"):<20} {txt}')
    else:
        print('[INFO] All samples correctly classified.')

elif results_all:
    print('\nPer-sample predictions not available.')
    print('(Likely using a modified feature_fusion that does not save annot/pred)')
    print('\nShowing F1 distribution across splits:\n')

    f1_by_size = defaultdict(list)
    for r in results_all:
        try:
            n = int(os.path.basename(r['_file']).split('_')[0])
            f1 = r.get('f1_test', 0) * 100
            if f1 > 0:
                f1_by_size[n].append(f1)
        except Exception:
            pass

    PAPER_REF = {50: 74.8, 100: 76.3, 250: 77.1}
    if f1_by_size:
        fig, ax = plt.subplots(figsize=(10, 5))
        sizes = sorted(f1_by_size.keys())
        bp = ax.boxplot([f1_by_size[s] for s in sizes], patch_artist=True,
                        labels=[str(s) for s in sizes])
        for patch, c in zip(bp['boxes'], ['#42a5f5','#66bb6a','#ffa726','#ef5350']):
            patch.set_facecolor(c); patch.set_alpha(0.7)
        paper_pts = [(sizes.index(s)+1, v) for s, v in PAPER_REF.items() if s in sizes]
        if paper_pts:
            ax.plot([p[0] for p in paper_pts], [p[1] for p in paper_pts],
                    'r--o', label='Paper (Table V)', zorder=5)
        ax.legend()
        ax.set_xlabel('Labeled Samples'); ax.set_ylabel('F1-Score (%)')
        ax.set_title('F1 Distribution Across Splits'); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        os.makedirs('figures', exist_ok=True)
        plt.savefig('figures/figure3_f1dist.png', dpi=150, bbox_inches='tight')
        plt.show()
        for s in sizes:
            fs = f1_by_size[s]
            ref = PAPER_REF.get(s)
            gap = f'{np.mean(fs)-ref:+.1f}' if ref else '—'
            print(f'n={s:<5} splits={len(fs)}  F1={np.mean(fs):.2f}±{np.std(fs):.2f}  '
                  f'paper={ref if ref else "—"}  gap={gap}')
else:
    print('No results found. Run Section 5 (experiments) first.')
    print(f'CWD: {os.getcwd()}')


In [ ]:
# ─── Figure 3 (Reverse): Most Consistently CORRECT Samples ─────────────────────
# Shows the 6 samples model ALWAYS got right across all 40 splits (error rate = 0%)
# Complement to Figure 3 — reveals what the GNN+CLIP fusion learns reliably.
import os, json, glob, re, math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Install ftfy if needed ──────────────────────────────────────────────────
try:
    import ftfy as _ftfy; _HAS_FTFY = True
except ImportError:
    import subprocess as _sp
    _sp.run(['pip', 'install', 'ftfy', '-q'], capture_output=True)
    try: import ftfy as _ftfy; _HAS_FTFY = True
    except: _HAS_FTFY = False

def _fix_text(t):
    if not isinstance(t, str): return t
    if _HAS_FTFY: t = _ftfy.fix_text(t)
    else:
        try: t = t.encode('cp1252').decode('utf-8')
        except: pass
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', t)

def _resolve_image(img_path, image_base):
    if not image_base or not img_path: return None
    p = img_path.replace('\\', '/')
    if p.startswith('data_image/'): p = p[len('data_image/'):]
    cand = os.path.join(image_base, p)
    if os.path.exists(cand): return cand
    hits = glob.glob(os.path.join(image_base, '**', os.path.basename(p)), recursive=True)
    return hits[0] if hits else None

# ── Config ───────────────────────────────────────────────────────────────────
EXP_ID = 100
EXP_PREFIX = f'{EXP_ID}_s'
SEARCH_ROOTS = [
    '/content/mm-class-for-disaster-data-with-gnn',
    '/content/drive/MyDrive/CrisisSummarization',
    '/content',
]

# ── Step 1: Always rebuild sample_totals / sample_wrongs (fast: just JSON) ──
print("Scanning result files for aggregated counts...")
_sample_wrongs = defaultdict(int)
_sample_totals = defaultdict(int)
_result_root   = None
_dev_jsonl     = None

# Mirror cell 28: search `results/**/*.json` (not runtime_log.json)
def _find_result_files_rev():
    for base in SEARCH_ROOTS:
        files = glob.glob(os.path.join(base, 'results', '**', '*.json'), recursive=True)
        if files:
            print(f"  Found {len(files)} JSON files under: {base}/results/")
            return base, files
    # Fallback: broad search
    files = glob.glob('/content/**/results/**/*.json', recursive=True)
    if files:
        print(f"  Fallback: found {len(files)} files")
        return '/content', files
    print("  [WARN] No result JSON files found.")
    return None, []

_result_root, _all_json = _find_result_files_rev()
print(f"  Sample paths (first 3):")
for p in sorted(_all_json)[:3]: print(f"    {p}")

_n_no_annot = 0
for rf in sorted(_all_json):
    # Filter to EXP_ID results only
    if EXP_PREFIX not in rf: continue
    content = open(rf).read().strip()
    if not content: continue
    try: data = json.loads(content)
    except: continue
    annot = data.get('annot', [])
    pred  = data.get('pred',  [])
    if not annot or not pred or len(annot) != len(pred):
        _n_no_annot += 1
        if _n_no_annot <= 2:
            print(f"  [no_annot] keys={list(data.keys())} path={rf[-60:]}")
        continue
    sdir = os.path.dirname(rf)
    for idx, (a, p) in enumerate(zip(annot, pred)):
        _sample_totals[idx] += 1
        if a != p: _sample_wrongs[idx] += 1
    if _dev_jsonl is None:
        cand = os.path.join(sdir, 'dev.jsonl')
        if os.path.exists(cand): _dev_jsonl = cand
print(f"  Loaded {len(_sample_totals)} samples from {len(_all_json)} files (no_annot={_n_no_annot})")

_sample_error_rate = {
    idx: _sample_wrongs[idx] / _sample_totals[idx]
    for idx in _sample_totals if _sample_totals[idx] > 0
}
MAX_SPLITS = max(_sample_totals.values()) if _sample_totals else 40
print(f"Loaded {len(_sample_error_rate)} samples | MAX_SPLITS={MAX_SPLITS}")

# ── Step 2: Reuse or reload test_samples ─────────────────────────────────────
if 'test_samples' in dir() and test_samples:
    print(f"Reusing test_samples ({len(test_samples)} rows)")
    _test_samples = test_samples
else:
    _test_samples = None
    if _dev_jsonl and os.path.exists(_dev_jsonl):
        _test_samples = [json.loads(l) for l in open(_dev_jsonl, encoding='utf-8')]
        print(f"Loaded test_samples from {_dev_jsonl}: {len(_test_samples)} rows")
    else:
        # Fallback: search for any dev.jsonl for this EXP_ID
        for root in SEARCH_ROOTS:
            hits = glob.glob(os.path.join(root, '**', f'{EXP_ID}_s0', '**', 'dev.jsonl'), recursive=True)
            if hits:
                _test_samples = [json.loads(l) for l in open(hits[0], encoding='utf-8')]
                print(f"Loaded test_samples from {hits[0]}: {len(_test_samples)} rows")
                break

# ── Step 3: Reuse or find IMAGE_BASE ─────────────────────────────────────────
if 'IMAGE_BASE' in dir() and IMAGE_BASE and os.path.isdir(IMAGE_BASE):
    print(f"Reusing IMAGE_BASE: {IMAGE_BASE}")
    _image_base = IMAGE_BASE
else:
    _image_base = None
    for root in SEARCH_ROOTS:
        cand = os.path.join(root, 'data', 'CrisisMMD_v2.0', 'data_image')
        if os.path.isdir(cand):
            _image_base = cand; break
        hits = glob.glob(os.path.join(root, '**', 'data_image'), recursive=True)
        if hits:
            _image_base = hits[0]; break
    print(f"IMAGE_BASE: {_image_base}")

# ── Step 4: Find top-N always-correct samples ─────────────────────────────────
N_SHOW = 6

always_correct = sorted(
    [(idx, rate) for idx, rate in _sample_error_rate.items()
     if _sample_totals[idx] == MAX_SPLITS and rate == 0.0],
    key=lambda x: x[0]
)

print(f"\nTotal ALWAYS-CORRECT samples (0% error, {MAX_SPLITS}/{MAX_SPLITS} splits): {len(always_correct)}")

if len(always_correct) < N_SHOW:
    print(f"[WARN] Fewer than {N_SHOW} perfect samples, using lowest error rate instead.")
    always_correct = sorted(_sample_error_rate.items(), key=lambda x: (x[1], x[0]))

top_correct = always_correct[:N_SHOW]

print(f"\n{'Rank':<6} {'Idx':<8} {'Correct/Total':<16} {'Correct%'}")
print("─" * 45)
for rank, (idx, rate) in enumerate(top_correct, 1):
    total = _sample_totals[idx]
    ncorr = total - _sample_wrongs.get(idx, 0)
    print(f"{rank:<6} {idx:<8} {ncorr}/{total:<12} {(1-rate)*100:.0f}%")

# ── Step 5: Plot ─────────────────────────────────────────────────────────────
n_rows = math.ceil(N_SHOW / 3)
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5.5 * n_rows))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

table_lines = []
table_lines.append(f"\n{'Rank':<6} {'Correct%':<10} {'Label':<22} {'Tweet text'}")
table_lines.append("─" * 90)

for rank, (idx, rate) in enumerate(top_correct):
    ax = axes[rank]
    correct_pct = (1 - rate) * 100

    sample    = (_test_samples[idx] if _test_samples and idx < len(_test_samples) else {})
    raw_label = sample.get('label', 'unknown')
    label     = 'informative' if raw_label in (1, '1', 'informative') else 'not_informative'
    tweet_txt = _fix_text(sample.get('tweet_text', sample.get('text', '')) or '')
    img_rel   = sample.get('image', sample.get('image_path', ''))
    img_path  = _resolve_image(img_rel, _image_base) if img_rel else None

    if img_path and os.path.exists(img_path):
        try:
            from matplotlib.image import imread
            ax.imshow(imread(img_path), aspect='auto')
        except Exception as e:
            ax.set_facecolor('#1e2a1e')
            ax.text(0.5, 0.5, f'Load error:\n{e}', ha='center', va='center',
                    color='#88ff88', fontsize=8, transform=ax.transAxes)
    else:
        ax.set_facecolor('#1e2a1e')
        ax.text(0.5, 0.5, '(Image not found)', ha='center', va='center',
                color='#555', fontsize=10, transform=ax.transAxes)

    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('#00e676'); spine.set_linewidth(3)

    label_color = '#64ffda' if label == 'informative' else '#ffab40'
    ax.set_title(f"Label: {label} | Correct Rate: ({correct_pct:.0f}%)",
                 color=label_color, fontsize=10, fontweight='bold', pad=6)

    import textwrap as _tw
    short = tweet_txt[:80].replace('\n', ' ')
    ax.set_xlabel('\n'.join(_tw.wrap(short, width=40)),
                  color='#cccccc', fontsize=8, fontstyle='italic', labelpad=8)

    table_lines.append(f"{rank+1:<6} {correct_pct:<9.0f}%  {label:<22} {tweet_txt[:60]}")

for j in range(N_SHOW, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    f"FIGURE 3 (Reverse) – Most Consistently Correct Instances from the Test Set of CrisisMMD\n"
    f"(Always correct across all {MAX_SPLITS} test folds — green border = model always right | {len(always_correct)} samples = {len(always_correct)/len(_sample_error_rate)*100:.1f}% of test set)",
    color='white', fontsize=12, fontweight='bold', y=0.02
)
plt.tight_layout(rect=[0, 0.06, 1, 1])
os.makedirs('figures', exist_ok=True)
out_path = 'figures/figure3_reverse_crisismmd.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"\nSaved: {out_path}")
print('\n'.join(table_lines))


In [ ]:
# ─── Figure 3 (Adjustable Rate): Samples at Any Target Error Rate ─────────────
# ════════════════════════════════════════════════════════════════════════════════
# CONFIGURE HERE:
TARGET_RATE = 0.5    # Target error rate  (0.0 = always correct, 1.0 = always wrong, 0.5 = coin flip)
TOLERANCE   = 0.15   # Pool: samples with |rate - TARGET_RATE| <= TOLERANCE are eligible
N_SHOW      = 6      # Number of samples to display (randomly drawn from pool)
RANDOM_SEED = None   # Set an int (e.g. 42) for reproducible picks, None = new each run
# ════════════════════════════════════════════════════════════════════════════════
import os, json, glob, re, math, textwrap
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ── ftfy ──────────────────────────────────────────────────────────────────────
try:
    import ftfy as _ftfy; _HAS_FTFY = True
except ImportError:
    import subprocess as _sp
    _sp.run(['pip', 'install', 'ftfy', '-q'], capture_output=True)
    try: import ftfy as _ftfy; _HAS_FTFY = True
    except: _HAS_FTFY = False

def _fix_text(t):
    if not isinstance(t, str): return t
    if _HAS_FTFY: t = _ftfy.fix_text(t)
    else:
        try: t = t.encode('cp1252').decode('utf-8')
        except: pass
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', t)

def _resolve_image(img_path, image_base):
    if not image_base or not img_path: return None
    p = img_path.replace('\\', '/')
    if p.startswith('data_image/'): p = p[len('data_image/'):]
    cand = os.path.join(image_base, p)
    if os.path.exists(cand): return cand
    hits = glob.glob(os.path.join(image_base, '**', os.path.basename(p)), recursive=True)
    return hits[0] if hits else None

# ── Config ────────────────────────────────────────────────────────────────────
EXP_ID     = 100
EXP_PREFIX = f'{EXP_ID}_s'
SEARCH_ROOTS = [
    '/content/mm-class-for-disaster-data-with-gnn',
    '/content/drive/MyDrive/CrisisSummarization',
    '/content',
]

# ── Step 1: Always rebuild sample counts ──────────────────────────────────────
_sample_wrongs = defaultdict(int)
_sample_totals = defaultdict(int)
_dev_jsonl     = None

for root in SEARCH_ROOTS:
    found = glob.glob(os.path.join(root, 'results', '**', '*.json'), recursive=True)
    if not found:
        found = glob.glob('/content/**/results/**/*.json', recursive=True)
    if found:
        for rf in sorted(found):
            if EXP_PREFIX not in rf: continue
            content = open(rf).read().strip()
            if not content: continue
            try: data = json.loads(content)
            except: continue
            annot = data.get('annot', [])
            pred  = data.get('pred',  [])
            if not annot or not pred or len(annot) != len(pred): continue
            for idx, (a, p) in enumerate(zip(annot, pred)):
                _sample_totals[idx] += 1
                if a != p: _sample_wrongs[idx] += 1
            if _dev_jsonl is None:
                cand = os.path.join(os.path.dirname(rf), 'dev.jsonl')
                if os.path.exists(cand): _dev_jsonl = cand
        break

_sample_error_rate = {
    idx: _sample_wrongs[idx] / _sample_totals[idx]
    for idx in _sample_totals if _sample_totals[idx] > 0
}
MAX_SPLITS = max(_sample_totals.values()) if _sample_totals else 10

# ── Step 2: test_samples ──────────────────────────────────────────────────────
if 'test_samples' in dir() and test_samples:
    _test_samples = test_samples
else:
    _test_samples = None
    if _dev_jsonl and os.path.exists(_dev_jsonl):
        _test_samples = [json.loads(l) for l in open(_dev_jsonl, encoding='utf-8')]
    else:
        for root in SEARCH_ROOTS:
            hits = glob.glob(os.path.join(root,'**',f'{EXP_ID}_s0','**','dev.jsonl'), recursive=True)
            if hits:
                _test_samples = [json.loads(l) for l in open(hits[0], encoding='utf-8')]
                break

# ── Step 3: IMAGE_BASE ────────────────────────────────────────────────────────
if 'IMAGE_BASE' in dir() and IMAGE_BASE and os.path.isdir(IMAGE_BASE):
    _image_base = IMAGE_BASE
else:
    _image_base = None
    for root in SEARCH_ROOTS:
        cand = os.path.join(root, 'data', 'CrisisMMD_v2.0', 'data_image')
        if os.path.isdir(cand): _image_base = cand; break
        hits = glob.glob(os.path.join(root,'**','data_image'), recursive=True)
        if hits: _image_base = hits[0]; break

# ── Step 4: Random sample from pool around TARGET_RATE ───────────────────────
import random as _random

# Pool: all samples with full split coverage AND within tolerance band
pool = [
    (idx, rate) for idx, rate in _sample_error_rate.items()
    if _sample_totals[idx] == MAX_SPLITS
    and abs(rate - TARGET_RATE) <= TOLERANCE
]

print(f"\nLoaded {len(_sample_error_rate)} samples | MAX_SPLITS={MAX_SPLITS}")
print(f"Target error rate : {TARGET_RATE*100:.0f}%  ±{TOLERANCE*100:.0f}%  →  "
      f"Pool size: {len(pool)} samples")

# Auto-widen tolerance if pool too small
_tol = TOLERANCE
while len(pool) < N_SHOW and _tol < 0.51:
    _tol = round(_tol + 0.05, 2)
    pool = [
        (idx, rate) for idx, rate in _sample_error_rate.items()
        if _sample_totals[idx] == MAX_SPLITS
        and abs(rate - TARGET_RATE) <= _tol
    ]
    if len(pool) >= N_SHOW:
        print(f"[INFO] Auto-widened tolerance to ±{_tol*100:.0f}% to get {len(pool)} candidates")

# Random draw from pool
_rng = _random.Random(RANDOM_SEED)
top_samples = _rng.sample(pool, min(N_SHOW, len(pool)))
# Sort display by actual rate for readability
top_samples.sort(key=lambda x: x[1])

actual_rates = [r for _, r in top_samples]
print(f"Randomly selected {len(top_samples)} samples"
      + (f" (seed={RANDOM_SEED})" if RANDOM_SEED is not None else " (seed=random)"))
print(f"\n{'Rank':<6} {'Idx':<8} {'Wrong/Total':<14} {'Error%':<10} {'Δ from target'}")
print("─" * 52)
for rank, (idx, rate) in enumerate(top_samples, 1):
    w = _sample_wrongs.get(idx, 0)
    t = _sample_totals[idx]
    print(f"{rank:<6} {idx:<8} {w}/{t:<11} {rate*100:<9.0f}%  Δ={abs(rate-TARGET_RATE)*100:.0f}%")

# ── Step 5: Plot ──────────────────────────────────────────────────────────────
n_rows = math.ceil(N_SHOW / 3)
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5.5 * n_rows))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for rank, (idx, rate) in enumerate(top_samples):
    ax = axes[rank]
    error_pct = rate * 100

    sample    = (_test_samples[idx] if _test_samples and idx < len(_test_samples) else {})
    raw_label = sample.get('label', 'unknown')
    label     = 'informative' if raw_label in (1,'1','informative') else 'not_informative'
    tweet_txt = _fix_text(sample.get('tweet_text', sample.get('text', '')) or '')
    img_rel   = sample.get('image', sample.get('image_path', ''))
    img_path  = _resolve_image(img_rel, _image_base) if img_rel else None

    if img_path and os.path.exists(img_path):
        try:
            from matplotlib.image import imread
            ax.imshow(imread(img_path), aspect='auto')
        except:
            ax.set_facecolor('#1a1a2e')
            ax.text(0.5, 0.5, '(Load error)', ha='center', va='center',
                    color='yellow', fontsize=9, transform=ax.transAxes)
    else:
        ax.set_facecolor('#1a1a2e')
        ax.text(0.5, 0.5, '(Image not found)', ha='center', va='center',
                color='#555', fontsize=10, transform=ax.transAxes)

    ax.set_xticks([]); ax.set_yticks([])

    # Border: interpolate red↔green based on error rate
    # 0% err → pure green, 50% → yellow, 100% → pure red
    r_ch = min(1.0, rate * 2)
    g_ch = min(1.0, (1 - rate) * 2)
    for spine in ax.spines.values():
        spine.set_edgecolor((r_ch, g_ch, 0.0))
        spine.set_linewidth(3.5)

    label_color = '#64ffda' if label == 'informative' else '#ffab40'
    dist_pct    = abs(rate - TARGET_RATE) * 100
    ax.set_title(
        f"Label: {label}\nError: {error_pct:.0f}%  (Δ from target: {dist_pct:.0f}%)",
        color=label_color, fontsize=9.5, fontweight='bold', pad=5
    )

    short = tweet_txt[:75].replace('\n', ' ')
    ax.set_xlabel('\n'.join(textwrap.wrap(short, width=42)),
                  color='#cccccc', fontsize=8, fontstyle='italic', labelpad=8)

for j in range(N_SHOW, len(axes)):
    axes[j].set_visible(False)

# Color legend
from matplotlib.patches import Patch
legend_patches = [
    Patch(color=(0.0, 1.0, 0.0), label='  0% error  — always correct'),
    Patch(color=(1.0, 1.0, 0.0), label='50% error  — coin flip'),
    Patch(color=(1.0, 0.0, 0.0), label='100% error — always wrong'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           framealpha=0.15, labelcolor='white', fontsize=9,
           bbox_to_anchor=(0.5, 0.01))

fig.suptitle(
    f"FIGURE 3 (Rate≈{TARGET_RATE*100:.0f}%) – {len(top_samples)} Random Samples from ±{_tol*100:.0f}% Band\n"
    f"Actual error rates: {min(actual_rates)*100:.0f}%–{max(actual_rates)*100:.0f}%  |  "
    f"Pool: {len(pool)} eligible samples  |  {MAX_SPLITS} test splits",
    color='white', fontsize=11, fontweight='bold', y=0.98
)
plt.tight_layout(rect=[0, 0.08, 1, 0.96])

os.makedirs('figures', exist_ok=True)
import time as _time
out_path = f'figures/figure3_rate{int(TARGET_RATE*100)}_crisismmd.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"\nSaved: {out_path}")
print(f"\nTo change target rate, set TARGET_RATE at the top of this cell.")
print(f"Examples: 0.0 = always correct | 0.5 = flip-flop | 1.0 = always wrong")


## 7. Experiment Report


In [ ]:
# ===============================================================
# 7. Save Full Experiment Report (auto-download)
# ===============================================================

import glob, datetime

# -- Collect result files --
result_files = sorted(glob.glob('results/**/*.json', recursive=True))
LOG.log_stage('Results Summary', {
    'num_result_files': len(result_files),
    'experiment_mode': EXPERIMENT_MODE,
})

# -- Save JSON report --
json_path = f'experiment_report_clip_pca.json'
report = LOG.save(json_path)

# -- Generate detailed Markdown report --
md_lines = []
md_lines.append('# Experiment Report')
md_lines.append(f'**Generated:** {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
md_lines.append(f'**Total runtime:** {report["total_time_s"]/60:.1f} minutes')
md_lines.append('')

# Metadata section
md_lines.append('## Experiment Configuration')
for k, v in report['metadata'].items():
    md_lines.append(f'- **{k}:** {v}')
md_lines.append('')

# Stages section
md_lines.append('## Execution Stages')
md_lines.append('| Stage | Time (min) | Details |')
md_lines.append('|-------|-----------|---------|')
for s in report['stages']:
    details = ', '.join(f'{k}={v}' for k,v in s.items() if k not in ('stage','timestamp','elapsed_s'))
    md_lines.append(f'| {s["stage"]} | {s["elapsed_s"]/60:.1f} | {details[:80]} |')
md_lines.append('')

# Results table (if aggregated results exist)
try:
    md_lines.append('## Results')
    md_lines.append('| Labeled | Splits | Our F1 (%) | Paper F1 (%) |')
    md_lines.append('|---------|--------|-----------|-------------|')
    for _, row in agg.iterrows():
        n = int(row['n_labeled'])
        f1 = f'{row["f1_mean"]*100:.1f} +/- {row["f1_std"]*100:.1f}'
# REMOVED: pf1 = paper_f1.get(n, '--')
        md_lines.append(f'| {n} | {int(row["n_splits"])} | {f1} | {pf1} |')
    md_lines.append('')
except:
    md_lines.append('*(Results not available - run Cell 6 first)*')
    md_lines.append('')

md_text = chr(10).join(md_lines)
md_path = f'experiment_report_clip_pca.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(md_text)
print(f'Markdown report saved: {md_path}')

# -- Display in notebook --
from IPython.display import Markdown, display
display(Markdown(md_text))

# -- Auto download --
try:
    from google.colab import files
    files.download(json_path)
    files.download(md_path)
    # Download individual result files too
    for rf in result_files[-5:]:
        files.download(rf)
    print(f'\nAuto-downloaded {2 + min(5, len(result_files))} files!')
except:
    print('Files saved. Download from Colab file browser (left panel).')


---

## Notes

### Experiment Modes

| Mode | Fusion | Reduction | EXP_ID |
|---|---|---|---|
| `clip_pca` | Late | PCA-256 | 100 |
| `clip_late` | Late | None | 101 |
| `clip_late_autoenc` | Late | Autoencoder | 102 |
| `clip_early` | Early | None | 110 |
| `clip_early_pca` | Early | PCA-256 | 111 |
| `clip_early_autoenc` | Early | Autoencoder | 112 |
| `clip_middle` | Middle | None | 120 |
| `clip_middle_pca` | Middle | PCA-256 | 121 |
| `clip_middle_autoenc` | Middle | Autoencoder | 122 |
| `maxvit_mpnet` | Late | None | 5 |

> **7Sept dataset** not publicly available (Twitter/X ToS). Tables 4b, 5, Figure 4 are paper reference values only.


## 8. Save All & Download


In [ ]:
# ═══════════════════════════════════════════════════════
# 8. SAVE ALL & DOWNLOAD — Run before closing browser!
# ═══════════════════════════════════════════════════════

import os, glob, zipfile, datetime

# --- Define all possible output files ---
file_patterns = [
    # Experiment results
    'results/**/*.json',
    'results/**/*.pkl',
    # Cached features (important! takes long to regenerate)
    'cached_features/*.pkl',
    'cached_features/*.npy',
    # Figures
    'figure*.png',
    '*.png',
    # Reports
    'experiment_report_*.json',
    'experiment_report_*.md',
    'experiment_log_*.json',
    # Data splits (important for reproducibility)
    'data/*.jsonl',
    'data/annotations/*.tsv',
    # Model checkpoints
    '*.pt', '*.pth',
]

# --- Collect existing files ---
found_files = set()
for pattern in file_patterns:
    for f in glob.glob(pattern, recursive=True):
        if os.path.isfile(f) and os.path.getsize(f) > 0:
            found_files.add(f)

# Also check for the notebook itself
if os.path.exists('mm_class_experiment.ipynb'):
    found_files.add('mm_class_experiment.ipynb')

found_files = sorted(found_files)

if not found_files:
    print("⚠️  No output files found. Run experiments first!")
else:
    # --- Create timestamped zip ---
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    zip_name = f'crisis_experiment_backup_{ts}.zip'
    
    total_size = 0
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in found_files:
            zf.write(f)
            size = os.path.getsize(f)
            total_size += size
            print(f"  ✅ {f} ({size/1024:.1f} KB)")
    
    zip_size = os.path.getsize(zip_name)
    print(f"\n{'='*50}")
    print(f"📦 {zip_name}")
    print(f"   Files: {len(found_files)}")
    print(f"   Original: {total_size/1024/1024:.1f} MB")
    print(f"   Compressed: {zip_size/1024/1024:.1f} MB")
    print(f"{'='*50}")
    
    # --- Auto download ---
    try:
        from google.colab import files
        files.download(zip_name)
        print(f"\n🔽 Download started: {zip_name}")
    except ImportError:
        print(f"\nNot on Colab. File saved: {zip_name}")
    except Exception as e:
        print(f"\nAuto-download failed: {e}")
        print(f"Download manually from Files panel (left sidebar).")

    # --- Summary by category ---
    categories = {
        'Cached Features': [f for f in found_files if 'cached_features' in f],
        'Results': [f for f in found_files if f.startswith('results/')],
        'Figures': [f for f in found_files if f.endswith('.png')],
        'Reports': [f for f in found_files if 'report' in f or 'log' in f],
        'Data': [f for f in found_files if f.startswith('data/')],
        'Other': [f for f in found_files if not any(
            x in f for x in ['cached_features', 'results/', '.png', 'report', 'log', 'data/']
        )],
    }
    print("\n📋 Summary:")
    for cat, files_list in categories.items():
        if files_list:
            cat_size = sum(os.path.getsize(f) for f in files_list)
            print(f"   {cat}: {len(files_list)} files ({cat_size/1024/1024:.1f} MB)")
